# CrewAI Agentic RAG — SEC Filings

Role-based agentic RAG using CrewAI for SEC 10-K/10-Q question answering.

**Agent Architecture (Sequential Crew):**
1. **QueryPlanner** — Decomposes questions into sub-queries with ticker/year/form_type hints
2. **FinancialResearcher** — Executes hybrid BM25 + dense retrieval via `hybrid_search` tool
3. **FinancialAnalyst** — Generates precise answers from retrieved context
4. **QualityReviewer** — Critiques and repairs the answer in a single combined step
5. **EvaluationJudge** — Scores candidate answers against golden references (eval only)

**Framework contrast vs LangGraph/LlamaIndex:**
- LangGraph: explicit state machine with conditional routing
- LlamaIndex: event-driven workflow with typed events
- CrewAI: role-based agents with delegated tasks, sequential process


In [ ]:
!pip uninstall -y chromadb chroma-hnswlib
!pip install --no-cache-dir chromadb

In [3]:
# ── Install dependencies (run once) ──────────────────────────────────────────
!pip install crewai crewai-tools rank_bm25 sentence-transformers #chromadb
!pip install langchain-groq langchain-google-genai python-dotenv tqdm pandas
!pip install litellm   # required by crewai LLM class

ERROR: Invalid requirement: '#chromadb': Expected package name at the start of dependency specifier
    #chromadb
    ^


ERROR: Invalid requirement: '#': Expected package name at the start of dependency specifier
    #
    ^


In [2]:
import os
import re
import json
import time
import warnings
from dataclasses import dataclass
from typing import Any, Dict, List, Optional, Literal, Type

import chromadb
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from dotenv import load_dotenv

from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer, CrossEncoder
from pydantic import BaseModel, Field, field_validator

from crewai import Agent, Task, Crew, Process, LLM
from crewai.tools import BaseTool

warnings.filterwarnings('ignore')
load_dotenv()
print('Libraries loaded.')

c:\Users\jolen\anaconda3\envs\genai_proj\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Libraries loaded.


In [3]:
CONFIG: Dict[str, Any] = {
    'random_seed': 42,

    # Pilot vs full run
    'use_pilot':             True,
    'pilot_n_questions':     10,
    'full_n_questions':      80,
    'use_llm_judge':         True,
    'use_few_shot_examples': True,
    'pilot_judge_sample_n':  1,
    'full_judge_sample_n':   2,

    # 'dev' (faster/cheaper) or 'final' (best quality)
    'execution_profile': 'dev',
    'provider':          os.getenv('LLM_PROVIDER', 'groq'),

    # Provider fallback order and per-provider RPM caps (matches langgraph_agentic_rag_sec_v3)
    'provider_fallback_order': ['groq', 'gemini'],
    'provider_rpm':            {'groq': 28, 'gemini': 10},

    # SEC dataset paths — update to match your environment
    'sec_chunks_path':   os.getenv('SEC_CHUNKS_PATH',   r'../sec_rag_team_share/sec_data/chunks/sec_chunks.jsonl'),
    'chroma_db_path':    os.getenv('CHROMA_DB_PATH',    r'../sec_rag_team_share/chroma_db'),
    'sec_eval_csv_path': os.getenv('SEC_EVAL_CSV_PATH', r'../sec_rag_team_share/evaluation/GenAI Eval QA.csv'),
    'results_dir':       os.getenv('RESULTS_DIR',       r'./results'),

    # Eval split config (matches LangGraph v3)
    'train_split':               'dev',
    'eval_split':                'test',
    'verbose_eval_logging':      True,
    'auto_export_results_input': True,

    # Retrieval knobs
    'bm25_top_k':                       int(os.getenv('BM25_TOP_K',                '8')),
    'dense_top_k':                      int(os.getenv('DENSE_TOP_K',               '8')),
    'rerank_top_k':                     int(os.getenv('RERANK_TOP_K',              '5')),
    'decomposition_top_k_per_subquery': int(os.getenv('DECOMP_TOP_K_PER_SUBQUERY', '4')),

    # Embedding / reranker (MUST match what was used to build ChromaDB — 768-dim)
    'dense_model_name':    'sentence-transformers/all-mpnet-base-v2',
    'reranker_model_name': 'cross-encoder/ms-marco-MiniLM-L-6-v2',

    # Groq model names
    'groq_dev_generator_model':   'llama-3.1-8b-instant',
    'groq_dev_agent_model':       'llama-3.1-8b-instant',
    'groq_dev_judge_model':       'llama-3.1-8b-instant',
    'groq_final_generator_model': 'meta-llama/llama-4-scout-17b-16e-instruct',
    'groq_final_agent_model':     'llama-3.1-8b-instant',
    'groq_final_judge_model':     'llama-3.1-8b-instant',

    # Groq fallback model lists (tried in order when primary hits rate limit)
    'groq_fallback_agent_models':     ['llama-3.1-8b-instant', 'qwen/qwen3-32b'],
    'groq_fallback_generator_models': ['llama-3.1-8b-instant', 'qwen/qwen3-32b'],
    'groq_fallback_judge_models':     ['llama-3.1-8b-instant', 'qwen/qwen3-32b'],

    # Gemini model names
    'gemini_dev_generator_model':   'gemini-2.0-flash-lite',
    'gemini_dev_agent_model':       'gemini-2.0-flash-lite',
    'gemini_dev_judge_model':       'gemini-2.0-flash-lite',
    'gemini_final_generator_model': 'gemini-2.0-flash',
    'gemini_final_agent_model':     'gemini-2.0-flash',
    'gemini_final_judge_model':     'gemini-2.0-flash',

    # Temperatures
    'temperature_planner':   0.0,
    'temperature_generator': 0.2,
    'temperature_critic':    0.0,
    'temperature_judge':     0.0,

    # Context window caps
    'generator_max_context_chunks': 8,
    'generator_max_context_chars':  12000,
    'control_max_context_chunks':   4,
    'control_max_context_chars':    6000,
    'judge_max_context_chunks':     3,
    'judge_max_context_chars':      4000,

    # Rate limiting / pacing
    'max_rpm':                  28,
    'inter_question_sleep_sec': 1.5,
    'llm_max_retries':          3,
    'llm_retry_base_delay_sec': 5,
}

CONFIG['judge_sample_n'] = (
    CONFIG['pilot_judge_sample_n'] if CONFIG['use_pilot'] else CONFIG['full_judge_sample_n']
) if CONFIG['use_llm_judge'] else 0

print(f'Config loaded. Provider: {CONFIG["provider"]}, Profile: {CONFIG["execution_profile"]}')


Config loaded. Provider: groq, Profile: dev


In [4]:
# ── Dataclass ─────────────────────────────────────────────────────────────────
@dataclass
class RetrievedChunk:
    doc_name:         str
    company:          str
    ticker:           str
    form_type:        str
    filing_year:      int
    page_num:         int
    chunk_id:         str
    raw_chunk:        str
    contextual_chunk: str
    score:            float
    source:           str


# ── Pydantic Schemas ──────────────────────────────────────────────────────────
class SubQuery(BaseModel):
    query:       Optional[str] = Field(default=None)
    ticker:      Optional[str] = Field(default=None)
    filing_year: Optional[int] = Field(default=None)
    form_type:   Optional[str] = Field(default=None)

    @field_validator('ticker', mode='before')
    @classmethod
    def _norm_ticker(cls, v):
        if v is None: return None
        return str(v).strip().upper() or None

    @field_validator('form_type', mode='before')
    @classmethod
    def _norm_form(cls, v):
        if v is None: return None
        c = str(v).strip().upper()
        return c if c in {'10-K', '10-Q'} else None


class PlannerOutput(BaseModel):
    needs_decomposition: bool = False
    sub_queries: List[SubQuery] = Field(default_factory=list)


class CriticRepairOutput(BaseModel):
    decision:     Literal['accept', 'repair', 'insufficient_data'] = 'accept'
    reason:       str = ''
    final_answer: str = ''


class JudgeOutput(BaseModel):
    score:          float = 0.0
    claims_covered: float = 0.0
    reason:         str   = ''


# ── Helpers ───────────────────────────────────────────────────────────────────
def safe_int(v, default: int = 0) -> int:
    try: return int(v)
    except: return default


def _rrf_score(rank: int, k: int = 60) -> float:
    return 1.0 / (k + rank + 1)


def format_retrieved_context(
    chunks: List[RetrievedChunk],
    max_chunks: int = 8,
    max_chars:  int = 12000,
) -> str:
    if not chunks:
        return 'No relevant context found.'
    parts, total = [], 0
    for i, c in enumerate(chunks[:max_chunks]):
        text = c.contextual_chunk or c.raw_chunk
        if total + len(text) > max_chars:
            rem = max_chars - total
            if rem > 100: text = text[:rem] + '...'
            else: break
        parts.append(f'[{i+1}] {text}')
        total += len(text)
    return '\n\n'.join(parts)


def extract_retrieved_doc_names(chunks: List[RetrievedChunk]) -> List[str]:
    seen: Dict[str, bool] = {}
    for c in chunks:
        if c.doc_name not in seen:
            seen[c.doc_name] = True
    return list(seen.keys())


def _try_parse_json(text: str, fallback: dict) -> dict:
    if not text: return fallback
    cleaned = re.sub(r'```(?:json)?\s*', '', text, flags=re.IGNORECASE).strip()
    cleaned = re.sub(r'```\s*$', '', cleaned).strip()
    try:
        return json.loads(cleaned)
    except json.JSONDecodeError:
        pass
    m = re.search(r'\{.*\}', cleaned, re.DOTALL)
    if m:
        try: return json.loads(m.group(0))
        except json.JSONDecodeError: pass
    return fallback


print('Schemas and helpers defined.')

Schemas and helpers defined.


In [5]:
class CorpusIndex:
    """Hybrid retrieval index: BM25 (sparse) + ChromaDB (dense) + CrossEncoder (rerank).
    Matches LangGraph v3 exactly — uses module-level dense_model and reranker globals.
    """

    def __init__(self, chunks: List[Dict[str, Any]], chroma_path: str):
        self.df = pd.DataFrame(chunks).copy()

        print('  Building BM25 index...')
        self.df['bm25_tokens'] = self.df['contextual_chunk'].str.lower().str.split()
        self.bm25 = BM25Okapi(self.df['bm25_tokens'].tolist())

        # Fast lookup: original chunk_id string → DataFrame row index
        self._str_to_row: Dict[str, int] = {
            row['chunk_id_str']: idx for idx, row in self.df.iterrows()
        }

        print('  Loading ChromaDB collection...')
        client = chromadb.PersistentClient(path=(chroma_path))
        collections = client.list_collections()
        if not collections:
            raise RuntimeError(f'No ChromaDB collections found at {chroma_path!r}')
        self.chroma_col = client.get_collection(collections[0].name)
        print(
            f'  ChromaDB ready: "{collections[0].name}"  '
            f'({self.chroma_col.count():,} docs)'
        )
        print(f'  CorpusIndex ready: {len(self.df):,} BM25 chunks + ChromaDB dense index')

    # ── Chunk constructors ────────────────────────────────────────────────────
    def _chunk_from_row(self, row_idx: int, score: float, source: str) -> RetrievedChunk:
        row = self.df.iloc[row_idx]
        return RetrievedChunk(
            doc_name=row['doc_name'],
            company=row['company'],
            ticker=row['ticker'],
            form_type=row['form_type'],
            filing_year=int(row['filing_year']),
            page_num=int(row.get('page_num', 0)),
            chunk_id=row['chunk_id_str'],
            raw_chunk=row['raw_chunk'],
            contextual_chunk=row['contextual_chunk'],
            score=float(score),
            source=source,
        )

    @staticmethod
    def _contextual_from_meta(text: str, meta: dict) -> str:
        return (
            f"Company: {meta['company_name']} ({meta['ticker']})\n"
            f"Filing: {meta['form_type']} | Date: {str(meta['filing_date'])[:10]} "
            f"| Year: {meta['filing_year']}\n"
            f"Section: {meta['section_title']}\n"
            f"Content: {text}"
        )

    def _chunk_from_chroma(
        self, doc_id: str, text: str, meta: dict, dist: float
    ) -> RetrievedChunk:
        doc_name = f"{meta['ticker']}_{meta['form_type']}_{str(meta['filing_date'])[:10]}"
        return RetrievedChunk(
            doc_name=doc_name,
            company=meta['company_name'],
            ticker=meta['ticker'],
            form_type=meta['form_type'],
            filing_year=int(meta['filing_year']),
            page_num=int(meta.get('chunk_index', 0)),
            chunk_id=doc_id,
            raw_chunk=text,
            contextual_chunk=self._contextual_from_meta(text, meta),
            score=1.0 - float(dist),
            source='chroma_dense',
        )

    # ── Filter helpers ────────────────────────────────────────────────────────
    def _bm25_mask(
        self, ticker: str = None, filing_year: int = None, form_type: str = None
    ) -> Optional[np.ndarray]:
        if not ticker and not filing_year and not form_type:
            return None
        mask = np.ones(len(self.df), dtype=float)
        if ticker:
            mask *= (self.df['ticker'].str.upper() == ticker.upper()).astype(float).values
        if filing_year:
            mask *= (self.df['filing_year'] == int(filing_year)).astype(float).values
        if form_type:
            mask *= (self.df['form_type'].str.upper() == form_type.upper()).astype(float).values
        if mask.sum() < 5:
            print('  [Warn] BM25 filter matched < 5 chunks — using full corpus')
            return None
        return mask

    def _chroma_where(
        self, ticker: str = None, filing_year: int = None, form_type: str = None
    ) -> Optional[dict]:
        conditions = []
        if ticker:      conditions.append({'ticker':      {'$eq': ticker.upper()}})
        if filing_year: conditions.append({'filing_year': {'$eq': int(filing_year)}})
        if form_type:   conditions.append({'form_type':   {'$eq': form_type.upper()}})
        if not conditions: return None
        return conditions[0] if len(conditions) == 1 else {'$and': conditions}

    # ── Search methods ────────────────────────────────────────────────────────
    def bm25_search(
        self, query: str, top_k: int, mask: Optional[np.ndarray] = None
    ) -> List[RetrievedChunk]:
        scores = np.array(self.bm25.get_scores(query.lower().split()))
        if mask is not None:
            scores = scores * mask
        top_idx = np.argsort(scores)[::-1]
        top_idx = [i for i in top_idx if scores[i] > 0][:top_k]
        return [self._chunk_from_row(i, scores[i], 'bm25') for i in top_idx]

    def dense_search(
        self,
        query: str,
        top_k: int,
        ticker: str = None,
        filing_year: int = None,
        form_type: str = None,
    ) -> List[RetrievedChunk]:
        # Uses module-level dense_model
        q_emb = dense_model.encode([query], normalize_embeddings=True)[0].tolist()
        where = self._chroma_where(ticker, filing_year, form_type)
        kwargs: dict = {
            'query_embeddings': [q_emb],
            'n_results': min(top_k, self.chroma_col.count()),
            'include': ['documents', 'metadatas', 'distances'],
        }
        if where:
            kwargs['where'] = where
        results = self.chroma_col.query(**kwargs)
        return [
            self._chunk_from_chroma(doc_id, text, meta, dist)
            for doc_id, text, meta, dist in zip(
                results['ids'][0],
                results['documents'][0],
                results['metadatas'][0],
                results['distances'][0],
            )
        ]

    def hybrid_search(
        self,
        query: str,
        bm25_top_k: int,
        dense_top_k: int,
        rerank_top_k: int,
        ticker: str = None,
        filing_year: int = None,
        form_type: str = None,
    ) -> List[RetrievedChunk]:
        mask         = self._bm25_mask(ticker, filing_year, form_type)
        bm25_results  = self.bm25_search(query, bm25_top_k, mask)
        dense_results = self.dense_search(query, dense_top_k, ticker, filing_year, form_type)

        rrf: Dict[str, float] = {}
        pool: Dict[str, RetrievedChunk] = {}
        for rank, chunk in enumerate(bm25_results):
            rrf[chunk.chunk_id]  = rrf.get(chunk.chunk_id, 0.0) + _rrf_score(rank)
            pool[chunk.chunk_id] = chunk
        for rank, chunk in enumerate(dense_results):
            rrf[chunk.chunk_id] = rrf.get(chunk.chunk_id, 0.0) + _rrf_score(rank)
            if chunk.chunk_id not in pool:
                pool[chunk.chunk_id] = chunk

        candidates = [pool[k] for k in sorted(rrf, key=rrf.__getitem__, reverse=True)]
        if not candidates:
            return []

        # Uses module-level reranker
        scores = reranker.predict(
            [(query, chunk.contextual_chunk) for chunk in candidates],
            show_progress_bar=False,
        )
        for chunk, score in zip(candidates, scores):
            chunk.score  = float(score)
            chunk.source = 'hybrid_reranked'
        return sorted(candidates, key=lambda c: c.score, reverse=True)[:rerank_top_k]


print('CorpusIndex defined.')

CorpusIndex defined.


In [6]:
# ── LLM Setup ─────────────────────────────────────────────────────────────────
provider = CONFIG['provider']
profile  = CONFIG['execution_profile']

if provider == 'groq':
    _gen_name   = CONFIG[f'groq_{profile}_generator_model']
    _agent_name = CONFIG[f'groq_{profile}_agent_model']
    _judge_name = CONFIG[f'groq_{profile}_judge_model']
    _key = os.getenv('GROQ_API_KEY', '')
    generator_llm = LLM(model=f'groq/{_gen_name}',   api_key=_key, temperature=CONFIG['temperature_generator'])
    agent_llm     = LLM(model=f'groq/{_agent_name}', api_key=_key, temperature=CONFIG['temperature_planner'])
    critic_llm    = LLM(model=f'groq/{_agent_name}', api_key=_key, temperature=CONFIG['temperature_critic'])
    judge_llm     = LLM(model=f'groq/{_judge_name}', api_key=_key, temperature=CONFIG['temperature_judge'])
else:
    _gen_name   = CONFIG[f'gemini_{profile}_generator_model']
    _agent_name = CONFIG[f'gemini_{profile}_agent_model']
    _judge_name = CONFIG[f'gemini_{profile}_judge_model']
    _key = os.getenv('GEMINI_API_KEY', '')
    generator_llm = LLM(model=f'gemini/{_gen_name}',   api_key=_key, temperature=CONFIG['temperature_generator'])
    agent_llm     = LLM(model=f'gemini/{_agent_name}', api_key=_key, temperature=CONFIG['temperature_planner'])
    critic_llm    = LLM(model=f'gemini/{_agent_name}', api_key=_key, temperature=CONFIG['temperature_critic'])
    judge_llm     = LLM(model=f'gemini/{_judge_name}', api_key=_key, temperature=CONFIG['temperature_judge'])

print(f'LLMs: generator={_gen_name}, agent={_agent_name}, judge={_judge_name}')

# ── Embedding + Reranker (module-level, used directly by CorpusIndex) ─────────
print('Loading embedding model...')
dense_model = SentenceTransformer(CONFIG['dense_model_name'])
print('Loading reranker...')
reranker = CrossEncoder(CONFIG['reranker_model_name'])
print('Models ready.')

LLMs: generator=llama-3.1-8b-instant, agent=llama-3.1-8b-instant, judge=llama-3.1-8b-instant
Loading embedding model...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3703.00it/s]
MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading reranker...


Loading weights: 100%|██████████| 105/105 [00:00<00:00, 3417.36it/s]
BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Models ready.


In [9]:
# ── Additional imports ────────────────────────────────────────────────────────
import threading
from collections import deque
from collections import Counter as _Counter
import re as _re

try:
    from langchain_groq import ChatGroq
    from langchain_google_genai import ChatGoogleGenerativeAI
    from langchain_core.prompts import ChatPromptTemplate
    _langchain_available = True
    print('LangChain judge backend: available')
except ImportError:
    _langchain_available = False
    print('[WARN] langchain_groq / langchain_google_genai not installed — LLM judge disabled')
    print('       pip install langchain-groq langchain-google-genai')


# ── Rate Limiter (matches langgraph_agentic_rag_sec_v3) ───────────────────────
class RateLimiter:
    """Sliding-window RPM throttle. Thread-safe."""
    def __init__(self, max_rpm: int):
        self.max_rpm = max_rpm
        self.window  = 60.0
        self._ts: deque = deque()
        self._lock = threading.Lock()

    def wait(self):
        with self._lock:
            now = time.time()
            while self._ts and now - self._ts[0] >= self.window:
                self._ts.popleft()
            if len(self._ts) >= self.max_rpm:
                sleep_for = self.window - (now - self._ts[0])
                if sleep_for > 0:
                    print(f'  [RateLimit] {self.max_rpm} RPM cap — waiting {sleep_for:.1f}s')
                    time.sleep(sleep_for)
            self._ts.append(time.time())


_rate_limiters: Dict[str, RateLimiter] = {
    provider: RateLimiter(max_rpm=rpm)
    for provider, rpm in CONFIG['provider_rpm'].items()
}

def get_rate_limiter(provider: str) -> RateLimiter:
    return _rate_limiters.get(provider, RateLimiter(max_rpm=CONFIG['max_rpm']))


# ── Error Detection ────────────────────────────────────────────────────────────
def is_rate_limit_error(exc: Exception) -> bool:
    msg = str(exc).lower()
    return 'rate limit' in msg or 'rate_limit' in msg or '429' in msg

def is_model_exhausted_error(exc: Exception) -> bool:
    msg = str(exc).lower()
    return any(m in msg for m in [
        'tokens per day', 'tpd', 'quota exceeded',
        'resource_exhausted', 'generatecontentinputtokens',
    ])


# ── Model Preference + Disable Tracking (matches langgraph_agentic_rag_sec_v3) ─
_preferred_models_by_task: Dict[str, str] = {}
_disabled_model_keys: set = set()

def disable_model_for_session(provider: str, model_name: str) -> None:
    key = f'{provider}::{model_name}'
    _disabled_model_keys.add(key)
    print(f'  [Model] Disabled {key} for this session.')

def get_task_model_snapshot(task_names: List[str]) -> Dict[str, str]:
    """Return current preferred model key for each task name."""
    return {t: _preferred_models_by_task.get(t, 'not_set') for t in task_names}


# ── Crew LLM Candidates (agent + generator pairs, for kickoff() fallback) ─────
def get_crew_model_candidates() -> List[tuple]:
    """
    Ordered list of (agent_model_name, gen_model_name, provider) tuples.
    Primary first, then within-provider fallbacks, then cross-provider.
    """
    candidates = []
    seen: set = set()
    profile = CONFIG['execution_profile']
    for provider in CONFIG.get('provider_fallback_order', [CONFIG['provider']]):
        primary_agent = CONFIG.get(f'{provider}_{profile}_agent_model', '')
        primary_gen   = CONFIG.get(f'{provider}_{profile}_generator_model', '')
        fb_agents = [primary_agent] + CONFIG.get(f'{provider}_fallback_agent_models', [])
        fb_gens   = [primary_gen]   + CONFIG.get(f'{provider}_fallback_generator_models', [])
        for a_model, g_model in zip(fb_agents, fb_gens):
            key = f'{provider}::{a_model}::{g_model}'
            if key not in seen and a_model and g_model:
                candidates.append((a_model, g_model, provider))
                seen.add(key)
    return candidates


# ── LangChain Judge LLM Candidates (for safe_invoke_structured) ───────────────
def get_judge_candidates() -> List[tuple]:
    """
    Ordered list of (langchain_llm, model_name, provider).
    Respects _preferred_models_by_task and _disabled_model_keys.
    """
    if not _langchain_available:
        return []
    profile = CONFIG['execution_profile']
    candidates = []
    seen: set = set()
    for provider in CONFIG.get('provider_fallback_order', [CONFIG['provider']]):
        primary  = CONFIG.get(f'{provider}_{profile}_judge_model', '')
        fallbacks = CONFIG.get(f'{provider}_fallback_judge_models', [])
        for model_name in [primary] + fallbacks:
            if not model_name:
                continue
            key = f'{provider}::{model_name}'
            if key in _disabled_model_keys or key in seen:
                continue
            seen.add(key)
            try:
                temp = CONFIG.get('temperature_judge', 0.0)
                if provider == 'groq':
                    llm = ChatGroq(
                        model=model_name,
                        api_key=os.getenv('GROQ_API_KEY', ''),
                        temperature=temp,
                    )
                elif provider == 'gemini':
                    llm = ChatGoogleGenerativeAI(
                        model=model_name,
                        google_api_key=os.getenv('GEMINI_API_KEY', ''),
                        temperature=temp,
                    )
                else:
                    continue
                candidates.append((llm, model_name, provider))
            except Exception as e:
                print(f'  [WARN] Could not create judge LLM {provider}:{model_name}: {e}')
    return candidates


# ── Judge Pydantic Schema ──────────────────────────────────────────────────────
class JudgeOutput(BaseModel):
    score:          float = Field(default=0.0, description='Score 0.0–1.0')
    claims_covered: float = Field(default=0.0, description='Fraction of key facts covered 0.0–1.0')
    reason:         str   = Field(default='',  description='Short explanation')


# ── Judge Prompts (verbatim from langgraph_agentic_rag_sec_v3) ────────────────
if _langchain_available:
    _judge_correctness_prompt = ChatPromptTemplate.from_messages([
        ('system',
         'Score the candidate answer against the reference answer from 0 to 1 for a financial QA task. '
         '1 = correct value, correct units, correct period. '
         '0 = wrong number, wrong company, wrong period, or completely missing. '
         'Give partial credit for answers close but with unit errors. '
         'Also set claims_covered: fraction of distinct facts/numbers in the reference present in the candidate. '
         'Return a valid JSON object only that matches the requested schema.'),
        ('human',
         'Question:\n{question}\n\nReference answer:\n{reference_answer}\n\nCandidate answer:\n{candidate_answer}'),
    ])
    _judge_faithfulness_prompt = ChatPromptTemplate.from_messages([
        ('system',
         'Score how well the candidate answer is grounded in the retrieved filing context from 0 to 1. '
         '1 = every number and claim is directly supported by the context. '
         '0 = answer contains numbers or claims not present in the context (hallucinated). '
         'Also set claims_covered: fraction of claims in the candidate supported by the context. '
         'Return a valid JSON object only that matches the requested schema.'),
        ('human',
         'Question:\n{question}\n\nRetrieved context:\n{context}\n\nCandidate answer:\n{candidate_answer}'),
    ])
else:
    _judge_correctness_prompt = None
    _judge_faithfulness_prompt = None


# ── Safe Judge Invoke (rate limiting + model fallback, matches langgraph v3) ───
def _safe_judge_invoke(prompt, variables: Dict[str, Any], task_name: str) -> JudgeOutput:
    """Invoke judge prompt with per-provider rate limiting and model fallback."""
    _fb = JudgeOutput(score=0.0, claims_covered=0.0, reason='judge fallback — all attempts failed')
    if not _langchain_available or prompt is None:
        return _fb
    candidates = get_judge_candidates()
    if not candidates:
        return _fb

    # Reorder: put preferred model first
    preferred = _preferred_models_by_task.get(task_name)
    if preferred:
        keys = [f'{p}::{m}' for _, m, p in candidates]
        if preferred in keys:
            idx = keys.index(preferred)
            candidates = candidates[idx:]

    max_retries = CONFIG['llm_max_retries']
    for model_idx, (llm, model_name, provider) in enumerate(candidates):
        for attempt in range(max_retries):
            try:
                get_rate_limiter(provider).wait()
                structured = llm.with_structured_output(JudgeOutput, method='json_mode')
                result = (prompt | structured).invoke(variables)
                _preferred_models_by_task[task_name] = f'{provider}::{model_name}'
                return result
            except Exception as e:
                print(f'  [WARN] Judge ({task_name}) attempt {attempt+1}/{max_retries} '
                      f'on {provider}:{model_name} failed: {e}')
                if is_model_exhausted_error(e):
                    disable_model_for_session(provider, model_name)
                    break
                if is_rate_limit_error(e) and model_idx < len(candidates) - 1:
                    next_llm, next_model, next_prov = candidates[model_idx + 1]
                    _preferred_models_by_task[task_name] = f'{next_prov}::{next_model}'
                    print(f'  [WARN] Switching judge to {next_prov}:{next_model}')
                    break
                if attempt < max_retries - 1:
                    delay = CONFIG['llm_retry_base_delay_sec'] * (2 ** attempt)
                    time.sleep(delay)
    return _fb


def llm_judge_correctness(
    question: str, reference_answer: str, candidate_answer: str,
) -> tuple:
    """
    Score candidate answer vs reference (correctness). Matches langgraph_agentic_rag_sec_v3.
    Returns: (score, claims_covered, reason, model_used)
    """
    result = _safe_judge_invoke(
        _judge_correctness_prompt,
        {'question': question, 'reference_answer': reference_answer, 'candidate_answer': candidate_answer},
        task_name='judge_correctness',
    )
    model_used = _preferred_models_by_task.get('judge_correctness', 'unknown')
    return (
        max(0.0, min(1.0, float(result.score))),
        max(0.0, min(1.0, float(result.claims_covered))),
        result.reason,
        model_used,
    )


def llm_judge_faithfulness(
    question: str, context: str, candidate_answer: str,
) -> tuple:
    """
    Score faithfulness to retrieved context. Matches langgraph_agentic_rag_sec_v3.
    Returns: (score, claims_covered, reason, model_used)
    """
    result = _safe_judge_invoke(
        _judge_faithfulness_prompt,
        {
            'question':         question,
            'context':          context[:CONFIG['judge_max_context_chars']],
            'candidate_answer': candidate_answer,
        },
        task_name='judge_faithfulness',
    )
    model_used = _preferred_models_by_task.get('judge_faithfulness', 'unknown')
    return (
        max(0.0, min(1.0, float(result.score))),
        max(0.0, min(1.0, float(result.claims_covered))),
        result.reason,
        model_used,
    )


# ── Scoring Utilities (from langgraph_agentic_rag_sec_v3) ─────────────────────
def _tokenize(text: str) -> List[str]:
    return _re.sub(r'[^\w\s]', '', text.lower()).split()

def token_f1_score(pred: str, ref: str) -> float:
    """Token-level F1 between predicted and reference answer."""
    if not pred or not ref:
        return 0.0
    pred_toks = _tokenize(pred)
    ref_toks  = _tokenize(ref)
    if not pred_toks or not ref_toks:
        return 0.0
    common = sum((_Counter(pred_toks) & _Counter(ref_toks)).values())
    if common == 0:
        return 0.0
    precision = common / len(pred_toks)
    recall    = common / len(ref_toks)
    return 2 * precision * recall / (precision + recall)

def numerical_accuracy_score(pred: str, ref: str) -> Optional[float]:
    """Fraction of reference numbers that appear in prediction (within 1% tolerance)."""
    nums_ref  = [float(n.replace(',', '')) for n in _re.findall(r'-?\d[\d,]*\.?\d*', ref)]
    nums_pred = [float(n.replace(',', '')) for n in _re.findall(r'-?\d[\d,]*\.?\d*', pred)]
    if not nums_ref:
        return None
    hits = sum(
        1 for r in nums_ref
        if any(abs(p - r) / (abs(r) + 1e-9) < 0.01 for p in nums_pred)
    )
    return hits / len(nums_ref)

def compute_decision_from_text(answer_text: str) -> str:
    """Detect whether the answer is a refusal or an actual answer."""
    lower = answer_text.lower()
    refusal_phrases = [
        'insufficient', 'not contain', 'not available', 'cannot find',
        "don't have", 'no information', 'not provided', 'unable to',
        'not enough', 'not present', 'not found', 'insufficient data',
    ]
    return 'refuse' if any(p in lower for p in refusal_phrases) else 'answer'


print('Rate limiter, model utilities, judge functions, and scoring utilities ready.')


LangChain judge backend: available
Rate limiter, model utilities, judge functions, and scoring utilities ready.


In [ ]:
# !pip uninstall -y chromadb chroma-hnswlib
# !pip install --no-cache-dir chromadb

In [10]:
# ── SEC Corpus Loading ────────────────────────────────────────────────────────

def load_sec_chunks(path: str) -> pd.DataFrame:
    """Load sec_chunks.jsonl and filter out low-value chunks."""
    chunks = []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                chunks.append(json.loads(line))
    df = pd.DataFrame(chunks)
    before = len(df)
    df = df[~df['flag_low_value_combined'].fillna(False)].reset_index(drop=True)
    if 'filing_year' in df.columns:
        df['filing_year'] = pd.to_numeric(df['filing_year'], errors='coerce').astype('Int64')
        filing_years = [int(v) for v in sorted(df['filing_year'].dropna().unique().tolist())]
    else:
        filing_years = []
    print(
        f'Loaded {before} total chunks → {len(df)} usable after filtering '
        f'({before - len(df)} low-value removed)'
    )
    print(
        f'Companies: {sorted(df["ticker"].dropna().astype(str).unique())}  |  '
        f'Filing years: {filing_years}'
    )
    return df


def sec_df_to_chunk_dicts(df: pd.DataFrame) -> List[Dict[str, Any]]:
    """Convert SEC DataFrame to the chunk dict format expected by CorpusIndex.
    Builds the contextual_chunk field by combining metadata + raw text.
    """
    chunks = []
    for i, row in df.iterrows():
        doc_name = f"{row['ticker']}_{row['form_type']}_{str(row['filing_date'])[:10]}"
        contextual_text = (
            f"Company: {row['company_name']} ({row['ticker']})\n"
            f"Filing: {row['form_type']} | Date: {str(row['filing_date'])[:10]} | Year: {row['filing_year']}\n"
            f"Section: {row['section_title']}\n"
            f"Content: {row['text']}"
        )
        chunks.append({
            'chunk_id':         i,                     # integer row index (BM25 lookup)
            'chunk_id_str':     str(row['chunk_id']),  # original string ID (ChromaDB cross-ref)
            'ticker':           row['ticker'],
            'company':          row['company_name'],
            'doc_name':         doc_name,
            'filing_year':      int(row['filing_year']),
            'form_type':        row['form_type'],
            'page_num':         int(row.get('chunk_index', 0)),
            'raw_chunk':        row['text'],
            'contextual_chunk': contextual_text,
        })
    return chunks


print('Loading SEC chunks...')
sec_df = load_sec_chunks(CONFIG['sec_chunks_path'])

print('Building CorpusIndex (BM25 + ChromaDB)...')
chunk_dicts  = sec_df_to_chunk_dicts(sec_df)
global_index = CorpusIndex(chunk_dicts, chroma_path=CONFIG['chroma_db_path'])
print('Done.')

Loading SEC chunks...
Loaded 13275 total chunks → 12725 usable after filtering (550 low-value removed)
Companies: ['AAPL', 'BAC', 'BRK-B', 'COST', 'CVX', 'JNJ', 'JPM', 'MSFT', 'NVDA', 'UNH', 'WMT', 'XOM']  |  Filing years: [2023, 2024, 2025]
Building CorpusIndex (BM25 + ChromaDB)...
  Building BM25 index...
  Loading ChromaDB collection...
  ChromaDB ready: "sec_filings"  (12,725 docs)
  CorpusIndex ready: 12,725 BM25 chunks + ChromaDB dense index
Done.


In [ ]:
# !pip uninstall -y chromadb chroma-hnswlib
# !pip install --no-cache-dir chromadb

Found existing installation: chromadb 1.1.1
Uninstalling chromadb-1.1.1:
  Successfully uninstalled chromadb-1.1.1


You can safely remove it manually.


   ---------------------------------------- 0.0/21.9 MB ? eta -:--:--
   ----------------- ---------------------- 9.7/21.9 MB 54.9 MB/s eta 0:00:01
   ---------------------------------------- 21.9/21.9 MB 57.7 MB/s  0:00:00


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
crewai 1.11.0 requires chromadb~=1.1.0, but you have chromadb 1.5.5 which is incompatible.


In [ ]:
# print('hello')

hello


In [ ]:
# import chromadb
# print("chromadb version:", chromadb.__version__)

# client = chromadb.PersistentClient(path="../sec_rag_team_share/chroma_db")
# print("client created")

# cols = client.list_collections()
# print("collections:", cols)

chromadb version: 1.5.5
client created
collections: [Collection(name=sec_filings)]


In [ ]:
# import chromadb
# from pathlib import Path

# test_path = Path("../sec_rag_team_share/chroma_db_fresh_test")
# test_path.mkdir(parents=True, exist_ok=True)

# client = chromadb.PersistentClient(path=str(test_path))
# print("fresh client OK")
# print(client.list_collections())

fresh client OK
[]


In [11]:
# ── Evaluation Data ───────────────────────────────────────────────────────────
# Mirrors LangGraph v3 evaluation setup.
# Falls back to hardcoded test questions if the CSV is not available.

FALLBACK_QUESTIONS = [
    {'id': 'FB_001', 'question': "What was NVIDIA's total revenue for fiscal year 2024?",
     'reference_answer': "NVIDIA's total revenue for FY2024 was $60.9 billion.",
     'question_type': 'extractive', 'company': 'NVIDIA', 'ticker': 'NVDA',
     'form_type': '10-K', 'filing_year': 2024},
    {'id': 'FB_002', 'question': "What were Apple's total net sales in fiscal year 2024?",
     'reference_answer': "Apple's total net sales in FY2024 were $391.0 billion.",
     'question_type': 'extractive', 'company': 'Apple', 'ticker': 'AAPL',
     'form_type': '10-K', 'filing_year': 2024},
    {'id': 'FB_003', 'question': "What was Microsoft's cloud revenue in FY2024?",
     'reference_answer': "Microsoft's Intelligent Cloud segment revenue was $105.4 billion in FY2024.",
     'question_type': 'extractive', 'company': 'Microsoft', 'ticker': 'MSFT',
     'form_type': '10-K', 'filing_year': 2024},
    {'id': 'FB_004', 'question': "What are NVIDIA's main risk factors as of their latest 10-K?",
     'reference_answer': "NVIDIA's risk factors include supply chain constraints, export controls, competition, and rapid technological change.",
     'question_type': 'qualitative', 'company': 'NVIDIA', 'ticker': 'NVDA',
     'form_type': '10-K', 'filing_year': 2024},
    {'id': 'FB_005', 'question': "Compare JPMorgan Chase and Bank of America's total assets in 2024.",
     'reference_answer': "JPMorgan Chase had total assets of approximately $3.9 trillion and Bank of America had approximately $3.3 trillion as of end of 2024.",
     'question_type': 'comparative', 'company': 'JPM/BAC', 'ticker': None,
     'form_type': '10-K', 'filing_year': 2024},
]

eval_df: Optional[pd.DataFrame] = None

try:
    raw_eval_df = pd.read_csv(CONFIG['sec_eval_csv_path'])
    raw_eval_df = raw_eval_df[raw_eval_df['question_id'].notna()].copy()
    raw_eval_df['question_id'] = raw_eval_df['question_id'].astype(str).str.strip()
    raw_eval_df = raw_eval_df[raw_eval_df['question_id'] != ''].copy()

    full_eval_df = raw_eval_df.rename(columns={
        'question_id':      'id',
        'expected_answer':  'reference_answer',
    }).copy()

    full_eval_df['id'] = full_eval_df['id'].apply(
        lambda x: f"SEC_{int(float(x)):03d}" if str(x).replace('.', '', 1).isdigit() else str(x)
    )
    for col in ('expected_decision', 'question_type', 'company', 'reference_answer', 'split'):
        if col in full_eval_df.columns:
            full_eval_df[col] = full_eval_df[col].fillna('').astype(str).str.strip()

    eval_source_df = full_eval_df[
        full_eval_df.get('split', pd.Series('', index=full_eval_df.index)) == CONFIG['eval_split']
    ].reset_index(drop=True)
    if eval_source_df.empty:
        eval_source_df = full_eval_df.reset_index(drop=True)

    n_sample = CONFIG['pilot_n_questions'] if CONFIG['use_pilot'] else CONFIG['full_n_questions']
    eval_df = eval_source_df.sample(
        n=min(n_sample, len(eval_source_df)),
        random_state=CONFIG['random_seed'],
    ).sort_values('id').reset_index(drop=True)

    print(f'Loaded eval CSV: {len(eval_df)} questions (split="{CONFIG["eval_split"]}")')
    print(eval_df[['id', 'question', 'reference_answer']].head(3).to_string(index=False))

except Exception as e:
    print(f'[Warn] Could not load eval CSV ({e}). Using fallback questions.')
    eval_df = pd.DataFrame(FALLBACK_QUESTIONS)
    print(f'Fallback eval set: {len(eval_df)} questions.')

Loaded eval CSV: 10 questions (split="test")
     id                                                                               question                                                                                                                                                  reference_answer
SEC_006                         What was Nvidia's gross margin percentage in fiscal year 2024?                                                                                               Nvidia's gross margin was 72.7% in FY2024, up from 56.9% in FY2023.
SEC_010                    What were Bank of America's Total liabilities as of September 2025?                                                                             Bank of America's total liabilities as of September 30, 2025 were $3,099,564 million.
SEC_016 What was UnitedHealth Group's medical care ratio for the three months ended June 2024? UnitedHealth Group's medical care ratio (MCR) for the three months ended June 30, 2024 wa

In [12]:
# ── Global Run State ──────────────────────────────────────────────────────────
# Module-level dict so HybridSearchTool can access the index and accumulate chunks.
_global_state: Dict[str, Any] = {
    'index':            global_index,
    'retrieved_chunks': [],
    'question':         '',
}


# ── CrewAI Tool: Hybrid Search ────────────────────────────────────────────────
class HybridSearchInput(BaseModel):
    query:       str           = Field(description='Search query optimized for SEC filing retrieval')
    ticker:      Optional[str] = Field(default=None, description='Company ticker e.g. NVDA, AAPL. Use when company-specific.')
    filing_year: Optional[int] = Field(default=None, description='Calendar year the filing was submitted e.g. 2023, 2024.')
    form_type:   Optional[str] = Field(default=None, description='10-K for annual, 10-Q for quarterly.')


class HybridSearchTool(BaseTool):
    name:        str             = 'hybrid_search'
    description: str             = (
        'Search SEC 10-K and 10-Q filings using hybrid BM25 + dense retrieval with '
        'cross-encoder reranking. Call once per sub-query from the planner. '
        'Provide ticker, filing_year, form_type when known to focus retrieval.'
    )
    args_schema: Type[BaseModel] = HybridSearchInput

    def _run(
        self,
        query:       str,
        ticker:      Optional[str] = None,
        filing_year: Optional[int] = None,
        form_type:   Optional[str] = None,
    ) -> str:
        index: CorpusIndex = _global_state.get('index')
        if index is None:
            return 'Error: No corpus index available.'

        print(f'  [HybridSearch] query={query!r:.60} ticker={ticker} year={filing_year} form={form_type}')

        # CorpusIndex.hybrid_search uses module-level dense_model and reranker
        chunks = index.hybrid_search(
            query=query,
            bm25_top_k=CONFIG['bm25_top_k'],
            dense_top_k=CONFIG['dense_top_k'],
            rerank_top_k=CONFIG['rerank_top_k'],
            ticker=ticker,
            filing_year=filing_year,
            form_type=form_type,
        )

        # Accumulate chunks (dedup by chunk_id)
        existing = _global_state.get('retrieved_chunks', [])
        seen_ids = {c.chunk_id for c in existing}
        for c in chunks:
            if c.chunk_id not in seen_ids:
                existing.append(c)
                seen_ids.add(c.chunk_id)
        _global_state['retrieved_chunks'] = existing

        print(f'  [HybridSearch] {len(chunks)} chunks returned (running total: {len(existing)})')
        # return format_retrieved_context(
        #     chunks,
        #     CONFIG['generator_max_context_chunks'],
        #     CONFIG['generator_max_context_chars'],
        # )

        return_val = "\n\n".join([
            f"[{c.ticker} {c.form_type} {c.filing_year}]\n{c.raw_chunk[:500]}"
            for c in chunks[:2]
        ])

        print(return_val)

        return "\n\n".join([
            f"[{c.ticker} {c.form_type} {c.filing_year}]\n{c.raw_chunk[:500]}"
            for c in chunks[:2]
        ])

search_tool = HybridSearchTool()
print('CrewAI tool defined.')

CrewAI tool defined.


In [15]:
# ── CrewAI Agents ─────────────────────────────────────────────────────────────
# Agents are built via _build_crew_agents() so build_and_run_crew() can
# recreate them with fallback LLMs without duplicating role/goal/backstory strings.

_COMPANIES = (
    'AAPL (Apple), BAC (Bank of America), BRK-B (Berkshire Hathaway), COST (Costco), '
    'CVX (Chevron), JNJ (Johnson & Johnson), JPM (JPMorgan Chase), MSFT (Microsoft), '
    'NVDA (NVIDIA), UNH (UnitedHealth), WMT (Walmart), XOM (ExxonMobil)'
)


def _build_crew_agents(agent_llm_obj, generator_llm_obj, critic_llm_obj):
    """
    Create fresh CrewAI Agent instances with the given LLM objects.
    Used for both primary setup and fallback retries.
    Returns: (planner, researcher, analyst, critic)
    """
    _verbose = CONFIG.get('verbose_eval_logging', True)

    planner = Agent(
        role='Query Decomposition Specialist',
        goal=(
            'Decompose complex financial questions into targeted sub-queries for SEC filing retrieval. '
            'Identify the companies, fiscal years, and filing types needed.'
        ),
        backstory=(
            f'You are a financial research planner specializing in SEC filings from: {_COMPANIES}. '
            'You analyze questions and determine whether they need data from multiple filings, '
            'then produce precise sub-queries with metadata hints (ticker, filing_year, form_type).'
        ),
        llm=agent_llm_obj,
        verbose=_verbose,
        allow_delegation=False,
        max_iter=2,
    )

    researcher = Agent(
        role='SEC Filing Research Specialist',
        goal=(
            'Retrieve the most relevant SEC filing passages to answer financial questions. '
            'Use the hybrid_search tool once per sub-query identified by the planner.'
        ),
        backstory=(
            'You are a financial researcher with deep expertise in SEC 10-K and 10-Q filings. '
            'You use hybrid BM25 + dense retrieval with cross-encoder reranking. '
            'When the planner identifies multiple sub-queries (comparing companies or years), '
            'you call hybrid_search separately for each, passing the metadata filters.'
        ),
        llm=agent_llm_obj,
        tools=[search_tool],
        verbose=_verbose,
        allow_delegation=False,
        max_iter=6,
    )

    analyst = Agent(
        role='Financial Analyst',
        goal=(
            'Generate precise, well-grounded answers from retrieved SEC filing context. '
            'Be exact with numbers, units (millions/billions), and fiscal periods.'
        ),
        backstory=(
            'You are a senior financial analyst who synthesizes data from SEC filings into clear answers. '
            'You provide precise figures with correct units and fiscal periods. '
            'You only answer from the provided context and explicitly state when data is missing. '
            'You never estimate or hallucinate financial figures.'
        ),
        llm=generator_llm_obj,
        verbose=_verbose,
        allow_delegation=False,
        max_iter=2,
    )

    critic = Agent(
        role='Financial QA Quality Reviewer',
        goal=(
            'Evaluate financial answers for accuracy and provide corrections when needed. '
            'Choose: accept (correct), repair (fix errors), or insufficient_data (data absent).'
        ),
        backstory=(
            'You are a meticulous financial QA reviewer. You verify answers against retrieved context, '
            'checking for correct numbers, units, fiscal periods, and line-item names. '
            'When you find errors, you immediately provide a corrected version. '
            'Use insufficient_data only when the required financial data is genuinely absent.'
        ),
        llm=critic_llm_obj,
        verbose=_verbose,
        allow_delegation=False,
        max_iter=2,
    )

    return planner, researcher, analyst, critic


# Primary instances (used on first attempt — no extra LLM object creation)
planner_agent, researcher_agent, analyst_agent, critic_agent = _build_crew_agents(
    agent_llm, generator_llm, critic_llm
)

# judge_agent kept for backward compatibility; judge is now done outside the crew
# via llm_judge_correctness / llm_judge_faithfulness (see rate-limiter cell above)
judge_agent = Agent(
    role='Financial QA Evaluation Judge',
    goal='Score candidate answers against reference answers on a 0–1 scale.',
    backstory=(
        'You are a strict financial QA judge who evaluates factual correctness: '
        'correct numbers, units, fiscal period, and company. '
        'Score 1.0 = fully correct, 0.5 = partial, 0.0 = wrong or no answer.'
    ),
    llm=judge_llm,
    verbose=False,
    allow_delegation=False,
    max_iter=2,
)

print('CrewAI agents defined.')


CrewAI agents defined.


In [13]:
# ── JSON Helper ───────────────────────────────────────────────────────────────
def _try_parse_json(raw: str, default: Any) -> Any:
    """Attempt to parse JSON from a raw string; return default on failure."""
    if not raw:
        return default
    # Strip markdown code fences if present
    text = raw.strip()
    for fence in ('```json', '```'):
        if text.startswith(fence):
            text = text[len(fence):]
    text = text.rstrip('`').strip()
    try:
        return json.loads(text)
    except Exception:
        pass
    # Fallback: extract first {...} block
    match = re.search(r'\{.*\}', text, re.DOTALL)
    if match:
        try:
            return json.loads(match.group())
        except Exception:
            pass
    return default


# ── Result Parser ─────────────────────────────────────────────────────────────
def _parse_crew_result(crew_result: Any, question: str) -> Dict[str, Any]:
    """Parse CrewAI crew output into a structured result dict (no in-crew judge)."""
    chunks = _global_state.get('retrieved_chunks', [])
    out: Dict[str, Any] = {
        'question':            question,
        'final_answer':        '',
        'generated_answer':    '',
        'critic_decision':     'accept',
        'critic_reason':       '',
        'repair_used':         False,
        'retrieved_chunks':    chunks,
        'retrieved_doc_names': extract_retrieved_doc_names(chunks),
        'needs_decomposition': False,
        'sub_queries':         [],
    }
    try:
        to = crew_result.tasks_output

        # Task 0 — Planner
        if len(to) > 0:
            pj = _try_parse_json(to[0].raw, {'needs_decomposition': False, 'sub_queries': []})
            out['needs_decomposition'] = bool(pj.get('needs_decomposition', False))
            out['sub_queries']         = pj.get('sub_queries', [])

        # Task 1 — Researcher (chunks captured via tool side-effect)

        # Task 2 — Analyst
        if len(to) > 2:
            out['generated_answer'] = to[2].raw.strip()
            out['final_answer']     = out['generated_answer']

        # Task 3 — Critic + Repair
        if len(to) > 3:
            cj = _try_parse_json(
                to[3].raw,
                {'decision': 'accept', 'reason': '', 'final_answer': out['generated_answer']},
            )
            out['critic_decision'] = cj.get('decision', 'accept')
            out['critic_reason']   = cj.get('reason', '')
            fa = cj.get('final_answer') or cj.get('revised_answer') or out['generated_answer']
            out['final_answer']    = str(fa).strip()
            out['repair_used']     = out['critic_decision'] == 'repair'

    except Exception as e:
        print(f'  [ParseResult] Warning: {e}')
    return out


# ── CrewAI LLM factory ────────────────────────────────────────────────────────
def _make_crewai_llm(model_name: str, provider: str, temperature: float) -> 'LLM':
    api_key = os.getenv('GROQ_API_KEY' if provider == 'groq' else 'GEMINI_API_KEY', '')
    return LLM(model=f'{provider}/{model_name}', api_key=api_key, temperature=temperature)


# ── Crew Builder with Rate Limiting + Model Fallback ─────────────────────────
def build_and_run_crew(
    question: str,
    golden_answer: Optional[str] = None,  # kept for API compat; judge is done separately
) -> Dict[str, Any]:
    """
    Build and run a sequential CrewAI crew for one question.

    Enhancements vs original:
    - Rate-limits kickoff() via per-provider RateLimiter (matches langgraph_agentic_rag_sec_v3)
    - Falls back to alternative models on rate-limit errors from kickoff()
    - Tracks which model was used for planner/generator/critic (model_planner, etc.)
    - Records latency_sec for the crew portion
    - Judge scoring is intentionally NOT done here — call llm_judge_correctness /
      llm_judge_faithfulness separately in the eval loop for proper rate limiting + fallback.
    """
    _global_state['retrieved_chunks'] = []
    _global_state['question']         = question

    start_time = time.time()

    # ── Task factory (same descriptions every attempt) ────────────────────────
    def _make_tasks(p_agent, r_agent, a_agent, c_agent) -> List:
        plan_task = Task(
            description=(
                f'Available companies: {_COMPANIES}.\n\n'
                f'Question: {question}\n\n'
                'Decide if this needs data from MULTIPLE distinct filings '
                '(different companies OR different fiscal years). Produce sub-queries.\n\n'
                'Rules:\n'
                '- Decompose only for 2+ companies or 2+ fiscal years.\n'
                '- Single-company/period: needs_decomposition=false, one sub-query.\n'
                '- filing_year = calendar year of submission '
                '(Apple FY2024 10-K filed Nov 2024 → filing_year=2024).\n\n'
                'Return ONLY valid JSON with keys: needs_decomposition (bool), '
                'sub_queries (list of: query, ticker, filing_year, form_type).'
            ),
            expected_output='Valid JSON: {"needs_decomposition": bool, "sub_queries": [...]}',
            agent=p_agent,
        )
        research_task = Task(
            description=(
                f'Research question: {question}\n\n'
                'Review the planner output. Call hybrid_search once per sub-query. '
                'Pass ticker, filing_year, form_type as filters when provided. '
                'If needs_decomposition=true, call once per sub-query separately.'
            ),
            expected_output='Comprehensive retrieval context with relevant SEC filing passages.',
            context=[plan_task],
            agent=r_agent,
        )
        generate_task = Task(
            description=(
                f'Question: {question}\n\n'
                'Using the retrieved context, provide a precise answer.\n'
                '- Quote exact figures (do not round unless source rounds).\n'
                '- Specify units (millions, billions, %).\n'
                '- Specify the fiscal period (FY2024, Q2 2023, etc.).\n'
                '- Name the exact line item from the filing.\n'
                '- If data is absent: "The retrieved context does not contain this information."'
            ),
            expected_output='Precise, well-grounded answer with exact figures, units, and periods.',
            context=[research_task],
            agent=a_agent,
        )
        critic_task = Task(
            description=(
                f'Question: {question}\n\n'
                'Evaluate the generated answer using the retrieved context. Check:\n'
                '1. Correct numbers and units (millions vs billions, sign).\n'
                '2. Correct fiscal period (FY2023 vs FY2024, Q1 vs Q2).\n'
                '3. Correct company and line-item names.\n'
                '4. Whether the answer is grounded in context.\n\n'
                'Choose ONE decision:\n'
                '- accept: factually correct and well-grounded.\n'
                '- repair: data present but answer has errors — provide corrected answer.\n'
                '- insufficient_data: required data genuinely absent from context.\n\n'
                'Return ONLY valid JSON: decision (string), reason (string), final_answer (string).\n'
                'final_answer: copy original if accept, corrected if repair, '
                '"Insufficient data: [reason]" if insufficient_data.'
            ),
            expected_output='Valid JSON: {"decision": str, "reason": str, "final_answer": str}',
            context=[research_task, generate_task],
            agent=c_agent,
        )
        return [plan_task, research_task, generate_task, critic_task]

    # ── Try crew with fallback models ─────────────────────────────────────────
    candidates  = get_crew_model_candidates()
    last_error: Optional[Exception] = None

    for attempt_idx, (agent_model, gen_model, provider) in enumerate(candidates):
        if attempt_idx > 0:
            print(f'  [CrewFallback] Attempt {attempt_idx + 1}: '
                  f'agent={provider}:{agent_model}  gen={provider}:{gen_model}')
            _global_state['retrieved_chunks'] = []  # reset before retry

        # Build CrewAI LLM objects for this attempt
        try:
            _agent_llm = _make_crewai_llm(agent_model, provider, CONFIG['temperature_planner'])
            _gen_llm   = _make_crewai_llm(gen_model,   provider, CONFIG['temperature_generator'])
            _crit_llm  = _make_crewai_llm(agent_model, provider, CONFIG['temperature_critic'])
        except Exception as e:
            print(f'  [CrewFallback] Could not build LLMs for {provider}:{agent_model}: {e}')
            last_error = e
            continue

        # Re-use primary global agents on first attempt (avoids unnecessary object creation)
        if attempt_idx == 0:
            p_a, r_a, a_a, c_a = planner_agent, researcher_agent, analyst_agent, critic_agent
        else:
            p_a, r_a, a_a, c_a = _build_crew_agents(_agent_llm, _gen_llm, _crit_llm)

        # Model log for this attempt
        model_log = {
            'model_planner':   f'{provider}::{agent_model}',
            'model_generator': f'{provider}::{gen_model}',
            'model_critic':    f'{provider}::{agent_model}',
        }

        tasks = _make_tasks(p_a, r_a, a_a, c_a)
        crew  = Crew(
            agents=[p_a, r_a, a_a, c_a],
            tasks=tasks,
            process=Process.sequential,
            verbose=CONFIG.get('verbose_eval_logging', True),
        )

        try:
            get_rate_limiter(provider).wait()
            crew_result = crew.kickoff()

            # Success — record preferred models
            _preferred_models_by_task['planner']   = model_log['model_planner']
            _preferred_models_by_task['generator'] = model_log['model_generator']
            _preferred_models_by_task['critic']    = model_log['model_critic']

            latency = time.time() - start_time
            result  = _parse_crew_result(crew_result, question)
            result.update({'latency_sec': latency, **model_log})
            return result

        except Exception as e:
            last_error = e
            print(f'  [Crew] Error on {provider}:{agent_model}: {e}')
            if is_rate_limit_error(e) and attempt_idx < len(candidates) - 1:
                print('  [CrewFallback] Rate limit — trying next candidate')
                continue
            break  # non-rate-limit error; stop retrying

    # ── All attempts failed ───────────────────────────────────────────────────
    latency = time.time() - start_time
    err_str = str(last_error) if last_error else 'unknown error'
    return {
        'question':            question,
        'final_answer':        f'Error: {err_str}',
        'generated_answer':    '',
        'critic_decision':     'error',
        'critic_reason':       err_str,
        'repair_used':         False,
        'retrieved_chunks':    [],
        'retrieved_doc_names': [],
        'needs_decomposition': False,
        'sub_queries':         [],
        'latency_sec':         latency,
        'model_planner':       'error',
        'model_generator':     'error',
        'model_critic':        'error',
        'error':               err_str,
    }


print('Crew builder defined.')


Crew builder defined.


In [18]:
CONFIG['bm25_top_k'] = 5
CONFIG['dense_top_k'] = 5
CONFIG['rerank_top_k'] = 3
CONFIG['generator_max_context_chunks'] = 3
CONFIG['generator_max_context_chars'] = 4000

In [16]:
# ── Single Question Smoke Test ────────────────────────────────────────────────
# test_q      = "What was NVIDIA's total revenue for fiscal year 2024?"
# test_golden = "NVIDIA's total revenue for fiscal year 2024 was $60.9 billion."


test_q = "What were Apple's total net sales in fiscal year 2024?"
test_golden =  "Apple's total net sales in FY2024 were $391.0 billion."

print(f'Question: {test_q}')
print('-' * 60)
result = build_and_run_crew(test_q, golden_answer=test_golden)

print()
print('=' * 60)
print(f'Final Answer:\n{result["final_answer"]}')
print(f'\nCritic:  {result["critic_decision"]} — {result["critic_reason"]}')
print(f'Repair:  {result["repair_used"]}')
if result.get('judge_score') is not None:
    print(f'Score:   {result["judge_score"]:.2f} | Claims: {result["judge_claims_covered"]:.2f}')
    print(f'Judge:   {result["judge_reason"]}')
print(f'Docs:    {result["retrieved_doc_names"]}')
print(f'Decomp:  {result["needs_decomposition"]} | Sub-queries: {len(result["sub_queries"])}')

Question: What were Apple's total net sales in fiscal year 2024?
------------------------------------------------------------


╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 848cd298-4be1-4a38-ba8b-7642194ba241                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Available companies: AAPL (Apple), BAC (Bank of America), BRK-B (Berkshire Hathaway), COST (Costco),     │
│  CVX (Chevron), JNJ (Johnson & Johnson), JPM (JPMorgan Chase), MSFT (Microsoft), NVDA (NVIDIA), UNH             │
│  (UnitedHealth), WMT (Walmart), XOM (ExxonMobil).                                                               │
│                                                                                                                 │
│  Question: What were Apple's total net sales in fiscal year 2024?                                               │
│                                                                                                                 │
│  Decide if this needs data from MULTIPLE distinct filings (different companies OR different fiscal years).      │
│  Produce sub-queries.                                                                                           │
│                                                                                                                 │
│  Rules:                                                                                                         │
│  - Decompose only for 2+ companies or 2+ fiscal years.                                                          │
│  - Single-company/period: needs_decomposition=false, one sub-query.                                             │
│  - filing_year = calendar year of submission (Apple FY2024 10-K filed Nov 2024 → filing_year=2024).             │
│                                                                                                                 │
│  Return ONLY valid JSON with keys: needs_decomposition (bool), sub_queries (list of: query, ticker,             │
│  filing_year, form_type).                                                                                       │
│  ID: 8fd04882-4d37-4fff-bb97-ed3f42f877fa                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Query Decomposition Specialist                                                                          │
│                                                                                                                 │
│  Task: Available companies: AAPL (Apple), BAC (Bank of America), BRK-B (Berkshire Hathaway), COST (Costco),     │
│  CVX (Chevron), JNJ (Johnson & Johnson), JPM (JPMorgan Chase), MSFT (Microsoft), NVDA (NVIDIA), UNH             │
│  (UnitedHealth), WMT (Walmart), XOM (ExxonMobil).                                                               │
│                                                                                                                 │
│  Question: What were Apple's total net sales in fiscal year 2024?                                               │
│                                                                                                                 │
│  Decide if this needs data from MULTIPLE distinct filings (different companies OR different fiscal years).      │
│  Produce sub-queries.                                                                                           │
│                                                                                                                 │
│  Rules:                                                                                                         │
│  - Decompose only for 2+ companies or 2+ fiscal years.                                                          │
│  - Single-company/period: needs_decomposition=false, one sub-query.                                             │
│  - filing_year = calendar year of submission (Apple FY2024 10-K filed Nov 2024 → filing_year=2024).             │
│                                                                                                                 │
│  Return ONLY valid JSON with keys: needs_decomposition (bool), sub_queries (list of: query, ticker,             │
│  filing_year, form_type).                                                                                       │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Query Decomposition Specialist                                                                          │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  {                                                                                                              │
│    "needs_decomposition": false,                                                                                │
│    "sub_queries": [                                                                                             │
│      {                                                                                                          │
│        "query": "SELECT total_net_sales FROM filings WHERE ticker = 'AAPL' AND filing_year = 2024 AND           │
│  form_type = '10-K'",                                                                                           │
│        "ticker": "AAPL",                                                                                        │
│        "filing_year": 2024,                                                                                     │
│        "form_type": "10-K"                                                                                      │
│      }                                                                                                          │
│    ]                                                                                                            │
│  }                                                                                                              │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Question: What were Apple's total net sales in fiscal year 2024?                                         │
│                                                                                                                 │
│  Evaluate the generated answer using the retrieved context. Check:                                              │
│  1. Correct numbers and units (millions vs billions, sign).                                                     │
│  2. Correct fiscal period (FY2023 vs FY2024, Q1 vs Q2).                                                         │
│  3. Correct company and line-item names.                                                                        │
│  4. Whether the answer is grounded in context.                                                                  │
│                                                                                                                 │
│  Choose ONE decision:                                                                                           │
│  - accept: factually correct and well-grounded.                                                                 │
│  - repair: data present but answer has errors — provide corrected answer.                                       │
│  - insufficient_data: required data genuinely absent from context.                                              │
│                                                                                                                 │
│  Return ONLY valid JSON: decision (string), reason (string), final_answer (string).                             │
│  final_answer: copy original if accept, corrected if repair, "Insufficient data: [reason]" if                   │
│  insufficient_data.                                                                                             │
│  Agent: Financial QA Quality Reviewer                                                                           │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[CrewAIEventsBus] Warning: Event pairing mismatch. 'crew_kickoff_completed' closed 'task_started' (expected 
'crew_kickoff_started')

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 848cd298-4be1-4a38-ba8b-7642194ba241                                                                       │
│  Final Output: {                                                                                                │
│    "decision": "accept",                                                                                        │
│    "reason": "The answer is factually correct and well-grounded in the context.",                               │
│    "final_answer": "Total net sales for Apple in fiscal year 2024 were $394.33 billion."                        │
│  }                                                                                                              │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


Final Answer:
Total net sales for Apple in fiscal year 2024 were $394.33 billion.

Critic:  accept — The answer is factually correct and well-grounded in the context.
Repair:  False
Docs:    ['AAPL_10-K_2024-11-01']
Decomp:  False | Sub-queries: 1


In [17]:
# ── Evaluation Loop (column schema matches langgraph_agentic_rag_sec_v3) ──────
def run_evaluation(df: pd.DataFrame) -> pd.DataFrame:
    """
    Run the full CrewAI evaluation loop.

    Per question:
      1. Run CrewAI crew (planner → researcher → analyst → critic) with rate limiting + fallback
      2. Run separate LLM judge for correctness (vs reference) and faithfulness (vs context)
      3. Compute heuristic scores: token_f1, numerical_accuracy, decision_accuracy

    Columns match langgraph_agentic_rag_sec_v3 for direct comparison.
    """
    all_results = []

    for i, row in tqdm(df.iterrows(), total=len(df), desc='CrewAI Eval'):
        # ── Question metadata (reads same CSV columns as langgraph_agentic_rag_sec_v3) ──
        question          = str(row.get('question', ''))
        reference_answer  = str(row.get('expected_answer', row.get('reference_answer', ''))).strip()
        question_id       = row.get('question_id', row.get('id', i))
        question_type     = row.get('question_type', 'unknown')
        company           = row.get('company',     '')
        ticker            = row.get('ticker',      '')
        form_type         = row.get('form_type',   '')
        filing_year       = row.get('filing_year', None)
        expected_decision = str(row.get('expected_decision', 'answer')).lower().strip()

        if CONFIG.get('verbose_eval_logging', True):
            print(f'\n{"=" * 60}')
            q_disp = question[:100] + '...' if len(question) > 100 else question
            print(f'Q{i + 1}/{len(df)} [{question_type}]  {ticker}  {filing_year}')
            print(f'  {q_disp}')

        # ── 1. Run CrewAI crew ─────────────────────────────────────────────────
        crew_result      = build_and_run_crew(question, golden_answer=None)
        final_answer     = crew_result.get('final_answer',     '')
        generated_answer = crew_result.get('generated_answer', '')
        retrieved_chunks = crew_result.get('retrieved_chunks', [])

        # ── 2. Heuristic scores ────────────────────────────────────────────────
        t_f1    = token_f1_score(final_answer, reference_answer) if reference_answer else None
        num_acc = numerical_accuracy_score(final_answer, reference_answer) if reference_answer else None
        predicted_decision = compute_decision_from_text(final_answer)
        decision_accuracy  = 1.0 if predicted_decision == expected_decision else 0.0

        # ── 3. Format context string for faithfulness judge ────────────────────
        context_str = format_retrieved_context(
            retrieved_chunks,
            max_chunks=CONFIG['judge_max_context_chunks'],
            max_chars=CONFIG['judge_max_context_chars'],
        )

        # ── 4. LLM Judge: Correctness ──────────────────────────────────────────
        llm_correctness_score = llm_claims_covered = llm_correctness_reason = None
        model_judge_correctness = 'skipped'
        if CONFIG.get('use_llm_judge', True) and reference_answer:
            c_score, c_covered, c_reason, c_model = llm_judge_correctness(
                question, reference_answer, final_answer,
            )
            llm_correctness_score   = c_score
            llm_claims_covered      = c_covered
            llm_correctness_reason  = c_reason
            model_judge_correctness = c_model

        # ── 5. LLM Judge: Faithfulness ─────────────────────────────────────────
        llm_faithfulness_score = llm_faithfulness_reason = None
        model_judge_faithfulness = 'skipped'
        if CONFIG.get('use_llm_judge', True):
            f_score, _, f_reason, f_model = llm_judge_faithfulness(
                question, context_str, final_answer,
            )
            llm_faithfulness_score   = f_score
            llm_faithfulness_reason  = f_reason
            model_judge_faithfulness = f_model

        # ── 6. Model usage snapshot ────────────────────────────────────────────
        model_snapshot = get_task_model_snapshot(
            ['planner', 'generator', 'critic', 'judge_correctness', 'judge_faithfulness']
        )

        # ── 7. Build result row ────────────────────────────────────────────────
        result_row = {
            # Question metadata
            'question_id':           question_id,
            'question':              question,
            'question_type':         question_type,
            'company':               company,
            'ticker':                ticker,
            'form_type':             form_type,
            'filing_year':           filing_year,
            'reference_answer':      reference_answer,
            'expected_decision':     expected_decision,
            # Answer
            'final_answer':          final_answer,
            'generated_answer':      generated_answer,
            'pipeline':              'crewai_agentic_rag',
            # Crew metadata
            'retrieved_doc_names':   ','.join(crew_result.get('retrieved_doc_names', [])),
            'needs_decomposition':   crew_result.get('needs_decomposition', False),
            'critic_decision':       crew_result.get('critic_decision', ''),
            'repair_used':           crew_result.get('repair_used', False),
            'latency_sec':           crew_result.get('latency_sec', None),
            # Heuristic scores
            'token_f1':              t_f1,
            'numerical_accuracy':    num_acc,
            'decision_accuracy':     decision_accuracy,
            'predicted_decision':    predicted_decision,
            # LLM Judge scores
            'llm_correctness_score': llm_correctness_score,
            'llm_claims_covered':    llm_claims_covered,
            'llm_correctness_reason':llm_correctness_reason,
            'llm_faithfulness_score':llm_faithfulness_score,
            'llm_faithfulness_reason':llm_faithfulness_reason,
            # Model logging (which model handled each step)
            'model_planner':             crew_result.get('model_planner',   model_snapshot.get('planner')),
            'model_generator':           crew_result.get('model_generator', model_snapshot.get('generator')),
            'model_critic':              crew_result.get('model_critic',    model_snapshot.get('critic')),
            'model_judge_correctness':   model_judge_correctness,
            'model_judge_faithfulness':  model_judge_faithfulness,
            # Raw chunks (dropped before CSV save)
            'retrieved_chunks':      retrieved_chunks,
        }
        all_results.append(result_row)

        if CONFIG.get('verbose_eval_logging', True):
            cd  = result_row['critic_decision']
            cs  = f"{llm_correctness_score:.2f}"  if llm_correctness_score  is not None else 'N/A'
            fs  = f"{llm_faithfulness_score:.2f}" if llm_faithfulness_score is not None else 'N/A'
            f1  = f"{t_f1:.2f}"                   if t_f1                  is not None else 'N/A'
            print(f'  → critic={cd}  corr={cs}  faith={fs}  f1={f1}')
            print(f'    planner={result_row["model_planner"]}  '
                  f'gen={result_row["model_generator"]}  '
                  f'judge_c={model_judge_correctness}')

        time.sleep(CONFIG['inter_question_sleep_sec'])

    return pd.DataFrame(all_results)


print(f'Running evaluation on {len(eval_df)} questions...')
results_df = run_evaluation(eval_df)
print(f'\nEvaluation complete. {len(results_df)} questions processed.')


Running evaluation on 10 questions...


CrewAI Eval:   0%|          | 0/10 [00:00<?, ?it/s]


Q1/10 [extractive]  NVDA  2024.0
  What was Nvidia's gross margin percentage in fiscal year 2024?


╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 5fd15180-6413-4a0f-b626-d5493a8ed65a                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Available companies: AAPL (Apple), BAC (Bank of America), BRK-B (Berkshire Hathaway), COST (Costco),     │
│  CVX (Chevron), JNJ (Johnson & Johnson), JPM (JPMorgan Chase), MSFT (Microsoft), NVDA (NVIDIA), UNH             │
│  (UnitedHealth), WMT (Walmart), XOM (ExxonMobil).                                                               │
│                                                                                                                 │
│  Question: What was Nvidia's gross margin percentage in fiscal year 2024?                                       │
│                                                                                                                 │
│  Decide if this needs data from MULTIPLE distinct filings (different companies OR different fiscal years).      │
│  Produce sub-queries.                                                                                           │
│                                                                                                                 │
│  Rules:                                                                                                         │
│  - Decompose only for 2+ companies or 2+ fiscal years.                                                          │
│  - Single-company/period: needs_decomposition=false, one sub-query.                                             │
│  - filing_year = calendar year of submission (Apple FY2024 10-K filed Nov 2024 → filing_year=2024).             │
│                                                                                                                 │
│  Return ONLY valid JSON with keys: needs_decomposition (bool), sub_queries (list of: query, ticker,             │
│  filing_year, form_type).                                                                                       │
│  ID: 6500fb5e-92a4-44b2-8734-7062c261d1fc                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

  [Crew] Error on groq:qwen/qwen3-32b: litellm.RateLimitError: RateLimitError: GroqException - {"error":{"message":"Rate limit reached for model `qwen/qwen3-32b` in organization `org_01kkmdjzb6fd8rxfjfvh041ns9` service tier `on_demand` on tokens per minute (TPM): Limit 6000, Used 5647, Requested 1154. Please try again in 8.01s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing","type":"tokens","code":"rate_limit_exceeded"}}

  [CrewFallback] Rate limit — trying next candidate
  [CrewFallback] Attempt 3: agent=gemini:gemini-2.0-flash-lite  gen=gemini:gemini-2.0-flash-lite
  [CrewFallback] Could not build LLMs for gemini:gemini-2.0-flash-lite: Error importing native provider: Authentication required. Provide one of:
  1. API key via GOOGLE_API_KEY or GEMINI_API_KEY environment variable
     (use_vertexai=True is optional for Vertex AI with API key)
  2. For Vertex AI with ADC: Set GOOGLE_CLOUD_PROJECT and run:
     gcloud auth application-default lo

╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

  [WARN] Could not create judge LLM gemini:gemini-2.0-flash-lite: 1 validation error for ChatGoogleGenerativeAI
  Value error, API key required for Gemini Developer API. Provide api_key parameter or set GOOGLE_API_KEY/GEMINI_API_KEY environment variable. [type=value_error, input_value={'model': 'gemini-2.0-fla...0.0, 'model_kwargs': {}}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.11/v/value_error
  → critic=error  corr=N/A  faith=0.00  f1=N/A
    planner=error  gen=error  judge_c=skipped


CrewAI Eval: 100%|██████████| 10/10 [01:38<00:00,  9.81s/it]


Evaluation complete. 10 questions processed.


In [23]:
results_df.question.iloc[0]

"What was Nvidia's gross margin percentage in fiscal year 2024?"

In [22]:
results_df.reference_answer.iloc[0]

"Nvidia's gross margin was 72.7% in FY2024, up from 56.9% in FY2023."

In [21]:
results_df.final_answer.iloc[0]

'Gross margin percentage for Nvidia in fiscal year 2024 was 69.4%.'

In [20]:
results_df

,question_id,question,question_type,company,ticker,form_type,filing_year,reference_answer,expected_decision,final_answer,...,llm_claims_covered,llm_correctness_reason,llm_faithfulness_score,llm_faithfulness_reason,model_planner,model_generator,model_critic,model_judge_correctness,model_judge_faithfulness,retrieved_chunks
0,SEC_006,What was Nvidia's gross margin percentage in f...,extractive,NVIDIA,NVDA,10-K,2024.0,"Nvidia's gross margin was 72.7% in FY2024, up ...",answer,Gross margin percentage for Nvidia in fiscal y...,...,0.5,,0.0,The candidate answer contains a number (69.4%)...,groq::llama-3.1-8b-instant,groq::llama-3.1-8b-instant,groq::llama-3.1-8b-instant,groq::llama-3.1-8b-instant,groq::llama-3.1-8b-instant,[RetrievedChunk(doc_name='NVDA_10-K_2025-02-26...
1,SEC_010,What were Bank of America's Total liabilities ...,extractive,Bank of America,BAC,10-Q,2025.0,Bank of America's total liabilities as of Sept...,answer,Bank of America's Total liabilities as of Sept...,...,0.9,,0.8,,groq::qwen/qwen3-32b,groq::qwen/qwen3-32b,groq::qwen/qwen3-32b,groq::llama-3.1-8b-instant,groq::llama-3.1-8b-instant,[RetrievedChunk(doc_name='BAC_10-Q_2025-10-31'...
2,SEC_016,What was UnitedHealth Group's medical care rat...,extractive,UnitedHealth Group,UNH,10-Q,2024.0,UnitedHealth Group's medical care ratio (MCR) ...,answer,Error: Error importing native provider: Authen...,...,0.0,,0.0,,error,error,error,groq::llama-3.1-8b-instant,groq::llama-3.1-8b-instant,[]
3,SEC_018,How many oil-equivalent barrels per day did Ex...,extractive,Exxon Mobil,XOM,10-K,2023.0,"ExxonMobil produced 3,738 thousand oil-equival...",answer,Error: Error importing native provider: Authen...,...,0.0,,0.0,,error,error,error,groq::llama-3.1-8b-instant,groq::llama-3.1-8b-instant,[]
4,SEC_024,What was Costco’s total revenue reported in fi...,extractive,Costco,COST,10-K,2025.0,Costco's total revenue for fiscal year 2025 wa...,answer,Error: Error importing native provider: Authen...,...,0.0,,0.0,,error,error,error,groq::llama-3.1-8b-instant,groq::llama-3.1-8b-instant,[]
5,SEC_033,What Common Equity Tier 1 capital ratio did JP...,paraphrased,JPMorgan,JPM,10-K,2023.0,JPMorgan's Standardized and Advanced CET1 capi...,answer,Error: Error importing native provider: Authen...,...,0.0,,0.0,,error,error,error,groq::llama-3.1-8b-instant,groq::llama-3.1-8b-instant,[]
6,SEC_039,How much revenue did Johnson & Johnson generat...,paraphrased,Johnson & Johnson,JNJ,10-Q,2025.0,Johnson & Johnson's total sales for the nine m...,answer,Error: Error importing native provider: Authen...,...,0.0,,0.0,,error,error,error,groq::llama-3.1-8b-instant,groq::llama-3.1-8b-instant,[]
7,SEC_041,What proportion of premium revenue did UnitedH...,paraphrased,UnitedHealth,UNH,10-Q,2024.0,UnitedHealth Group's medical care ratio (MCR) ...,answer,Error: Error importing native provider: Authen...,...,0.0,,0.0,,error,error,error,groq::llama-3.1-8b-instant,groq::llama-3.1-8b-instant,[]
8,SEC_042,What was the total profit that UnitedHealth Gr...,paraphrased,UnitedHealth,UNH,10-K,2023.0,UnitedHealth Group's net earnings reported in ...,answer,Error: Error importing native provider: Authen...,...,0.0,,0.0,,error,error,error,groq::llama-3.1-8b-instant,groq::llama-3.1-8b-instant,[]
9,SEC_091,What internal AI system does UnitedHealth use ...,adversarial,UnitedHealth,UNH,NaN,NaN,,refuse,Error: Error importing native provider: Authen...,...,NaN,NaN,0.0,,error,error,error,skipped,groq::llama-3.1-8b-instant,[]


In [19]:
results_df.to_csv('./sample_10_ans.csv', index=False)

In [ ]:
# ── Results & Metrics (matches langgraph_agentic_rag_sec_v3) ──────────────────
print('=' * 60)
print(' CREWAI AGENTIC RAG — EVALUATION SUMMARY')
print('=' * 60)

scored_c = results_df[results_df['llm_correctness_score'].notna()].copy()
scored_f = results_df[results_df['llm_faithfulness_score'].notna()].copy()

print(f'\nTotal questions       : {len(results_df)}')
print(f'Correctness judged    : {len(scored_c)}')
print(f'Faithfulness judged   : {len(scored_f)}')

# ── Overall metrics ───────────────────────────────────────────────────────────
print('\nOverall metrics:')
for col, label in [
    ('llm_correctness_score',  'LLM Correctness  '),
    ('llm_claims_covered',     'Claims Covered   '),
    ('llm_faithfulness_score', 'LLM Faithfulness '),
    ('token_f1',               'Token F1         '),
    ('numerical_accuracy',     'Numerical Accuracy'),
    ('decision_accuracy',      'Decision Accuracy'),
]:
    vals = results_df[col].dropna()
    if len(vals):
        print(f'  {label}: {vals.mean():.4f}  (n={len(vals)})')

# ── By question type ──────────────────────────────────────────────────────────
print('\nBy question_type:')
type_cols = ['llm_correctness_score', 'llm_faithfulness_score', 'token_f1', 'decision_accuracy']
avail_cols = [c for c in type_cols if c in results_df.columns]
if avail_cols and 'question_type' in results_df.columns:
    by_type = results_df.groupby('question_type')[avail_cols].mean().round(3)
    print(by_type.to_string())

# ── Critic decisions ──────────────────────────────────────────────────────────
print('\nCritic decision distribution:')
print(results_df['critic_decision'].value_counts().to_string())
print(f'\nRepair rate          : {results_df["repair_used"].mean():.1%}')
print(f'Decomposition rate   : {results_df["needs_decomposition"].mean():.1%}')

# ── Decision accuracy ─────────────────────────────────────────────────────────
if 'decision_accuracy' in results_df.columns:
    print(f'\nDecision accuracy    : {results_df["decision_accuracy"].mean():.1%}')
    print('  Predicted distribution:')
    print(results_df['predicted_decision'].value_counts().to_string())

# ── Model usage log ───────────────────────────────────────────────────────────
print('\nModel usage (most common per step):')
for col in ['model_planner', 'model_generator', 'model_critic',
            'model_judge_correctness', 'model_judge_faithfulness']:
    if col in results_df.columns and results_df[col].notna().any():
        top = results_df[col].value_counts().index[0]
        cnt = results_df[col].value_counts().iloc[0]
        print(f'  {col:28s}: {top}  ({cnt}/{len(results_df)})')

# ── Latency ───────────────────────────────────────────────────────────────────
if 'latency_sec' in results_df.columns:
    lat = results_df['latency_sec'].dropna()
    if len(lat):
        print(f'\nCrew latency: mean={lat.mean():.1f}s  '
              f'median={lat.median():.1f}s  max={lat.max():.1f}s')

# ── Score distribution ────────────────────────────────────────────────────────
if not scored_c.empty:
    print('\nCorrectness score distribution:')
    bins   = [0, 0.2, 0.4, 0.6, 0.8, 1.01]
    labels = ['0.0–0.2', '0.2–0.4', '0.4–0.6', '0.6–0.8', '0.8–1.0']
    scored_c['score_bin'] = pd.cut(
        scored_c['llm_correctness_score'], bins=bins, labels=labels, right=False
    )
    print(scored_c['score_bin'].value_counts().sort_index().to_string())

# ── Save results ──────────────────────────────────────────────────────────────
os.makedirs(CONFIG['results_dir'], exist_ok=True)
out_path = os.path.join(CONFIG['results_dir'], 'crewai_eval_results.csv')
results_df.drop(columns=['retrieved_chunks'], errors='ignore').to_csv(out_path, index=False)
print(f'\nResults saved → {out_path}')

# ── Preview ───────────────────────────────────────────────────────────────────
preview_cols = [
    'question_id', 'question_type', 'ticker', 'critic_decision',
    'llm_correctness_score', 'llm_faithfulness_score', 'token_f1',
    'model_planner', 'model_generator',
]
display_cols = [c for c in preview_cols if c in results_df.columns]
results_df[display_cols].head(10)
